# 🔍 Exploración del Dataset MagnaTagATune (MTAT)

> **Paper original:** Law et al., *Evaluation of Algorithms Using Games: The Case of Music Tagging* (ISMIR 2009)  
> **Usado por SemiSupCon** como dataset supervisado principal ($\mathcal{S}$).

MagnaTagATune es un dataset con:
- ~25,863 clips de audio (MP3, ~29s cada uno)
- 188 tags binarios anotados por humanos via un juego online
- Top 50 tags usados como benchmark estándar en MIR

---

## 0. Setup

In [ ]:
import os
import warnings
import subprocess
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import librosa
import librosa.display
from IPython.display import Audio, display, HTML

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.facecolor': '#f8f9fa',
    'axes.facecolor': '#ffffff',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})

# ── Montar Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/my_paper_data')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)

DATA_DIR = DRIVE_BASE
MTAT_DIR = DATA_DIR / 'magnatagatune'
MTAT_AUDIO = MTAT_DIR / 'audio'
MTAT_DIR.mkdir(exist_ok=True)

SR = 22050

def download_file(url, dest, desc=None):
    dest = Path(dest)
    if dest.exists():
        print(f'  ✓ Ya existe: {dest.name}')
        return
    print(f'  ↓ Descargando {desc or dest.name}...')
    def _p(bn, bs, ts):
        if ts > 0:
            print(f'\r    {min(100, bn*bs*100/ts):5.1f}% ({bn*bs/1024**2:.0f}/{ts/1024**2:.0f} MB)', end='', flush=True)
    urllib.request.urlretrieve(url, str(dest), _p)
    print(f'\n  ✓ Listo: {dest.name}')

print('Setup completo ✓')

## 1. Descarga del Dataset

MagnaTagATune se distribuye como:
- `mp3.zip.001`, `mp3.zip.002`, `mp3.zip.003` — Audio split en 3 partes (~200 MB c/u)
- `annotations_final.csv` — Tags binarios para cada clip
- `clip_info_final.csv` — Metadata adicional (artista, álbum, etc.)

In [ ]:
BASE_URL = 'https://mirg.city.ac.uk/datasets/magnatagatune'

# ── Descargar anotaciones ──
print('=== Anotaciones ===')
download_file(f'{BASE_URL}/annotations_final.csv',
              MTAT_DIR / 'annotations_final.csv',
              'annotations_final.csv (~1.8 MB)')

download_file(f'{BASE_URL}/clip_info_final.csv',
              MTAT_DIR / 'clip_info_final.csv',
              'clip_info_final.csv (~1.5 MB)')

# ── Descargar audio (3 partes) ──
print('\n=== Audio ===')
for i in range(1, 4):
    fname = f'mp3.zip.{i:03d}'
    download_file(f'{BASE_URL}/{fname}',
                  MTAT_DIR / fname,
                  f'{fname} (~200 MB)')

# ── Merge y extraer audio si no existe ──
if not MTAT_AUDIO.exists() or len(list(MTAT_AUDIO.rglob('*.mp3'))) < 100:
    combined = MTAT_DIR / 'mp3_all.zip'
    if not combined.exists():
        print('\n  📦 Combinando partes zip...')
        with open(str(combined), 'wb') as outf:
            for i in range(1, 4):
                part = MTAT_DIR / f'mp3.zip.{i:03d}'
                with open(str(part), 'rb') as inf:
                    outf.write(inf.read())
        print('  ✓ Combinado')
    
    print('  📦 Extrayendo audio...')
    MTAT_AUDIO.mkdir(exist_ok=True)
    subprocess.run(['unzip', '-qo', str(combined), '-d', str(MTAT_AUDIO)],
                   check=True)
    print('  ✓ Audio extraído')
else:
    print('\n  ✓ Audio ya extraído')

# Contar archivos
mp3_files = list(MTAT_AUDIO.rglob('*.mp3'))
print(f'\nTotal archivos MP3: {len(mp3_files):,}')

## 2. Exploración de Anotaciones (`annotations_final.csv`)

In [ ]:
annotations = pd.read_csv(MTAT_DIR / 'annotations_final.csv', sep='\t')

print(f'Dimensiones: {annotations.shape}')
print(f'\nColumnas totales: {len(annotations.columns)}')

# Separar tags de la columna de path
# La última columna es 'mp3_path', el resto son tags binarios
tag_columns = [c for c in annotations.columns if c != 'mp3_path']
print(f'Tags (columnas binarias): {len(tag_columns)}')
print(f'Clips: {len(annotations):,}')

display(annotations.head(10))

In [ ]:
# ── Distribución de tags ──
tag_sums = annotations[tag_columns].sum().sort_values(ascending=False)

print(f'Total de anotaciones tag=1: {tag_sums.sum():,}')
print(f'Promedio tags=1 por clip: {annotations[tag_columns].sum(axis=1).mean():.2f}')
print(f'\nTags con al menos 1 clip: {(tag_sums > 0).sum()}')
print(f'Tags con ≥ 50 clips: {(tag_sums >= 50).sum()}')
print(f'Tags con ≥ 100 clips: {(tag_sums >= 100).sum()}')

print(f'\n=== Top 50 Tags más frecuentes ===')
top50 = tag_sums.head(50)
for i, (tag, count) in enumerate(top50.items(), 1):
    bar = '█' * int(count / top50.max() * 30)
    print(f'  {i:2d}. {tag:25s} {count:>5,} {bar}')

In [ ]:
# ── Visualización: Top 50 tags ──
top50_tags = tag_sums.head(50)

fig, ax = plt.subplots(figsize=(14, 10))

colors = plt.cm.viridis(np.linspace(0.2, 0.9, 50))
bars = ax.barh(range(50), top50_tags.values, color=colors, edgecolor='white', height=0.75)
ax.set_yticks(range(50))
ax.set_yticklabels(top50_tags.index, fontsize=9)
ax.invert_yaxis()

for bar, val in zip(bars, top50_tags.values):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=8)

ax.set_xlabel('Número de Clips con este Tag')
ax.set_title('Top 50 Tags — MagnaTagATune', fontweight='bold', fontsize=14)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ── Distribución de tags por clip ──
tags_per_clip = annotations[tag_columns].sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(tags_per_clip, bins=30, color='#6C5CE7', edgecolor='white', alpha=0.85)
axes[0].axvline(tags_per_clip.median(), color='#E17055', linestyle='--', linewidth=2,
               label=f'Mediana: {tags_per_clip.median():.0f}')
axes[0].set_xlabel('Número de Tags por Clip')
axes[0].set_ylabel('Número de Clips')
axes[0].set_title('Tags por Clip', fontweight='bold')
axes[0].legend()
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Long tail de tags
axes[1].plot(range(len(tag_sums)), tag_sums.values, color='#00B894', linewidth=2)
axes[1].fill_between(range(len(tag_sums)), tag_sums.values, alpha=0.2, color='#00B894')
axes[1].axvline(50, color='red', linestyle='--', alpha=0.7, label='Top 50 cutoff')
axes[1].set_xlabel('Tag (ordenado por frecuencia)')
axes[1].set_ylabel('Número de Clips')
axes[1].set_title('Long Tail: Frecuencia de Tags', fontweight='bold')
axes[1].set_yscale('log')
axes[1].legend()
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print(f'Tags/clip — media: {tags_per_clip.mean():.2f}, mediana: {tags_per_clip.median():.0f}, max: {tags_per_clip.max()}')
print(f'Clips sin ningún tag: {(tags_per_clip == 0).sum():,}')

## 3. Co-ocurrencia de Tags

¿Qué tags aparecen juntos frecuentemente? Importante para entender la **contrastive matrix** en SemiSupCon.

In [ ]:
# ── Matriz de co-ocurrencia (Top 20 tags) ──
top20_names = tag_sums.head(20).index.tolist()
top20_data = annotations[top20_names]

cooccurrence = top20_data.T.dot(top20_data)

# Normalizar por la diagonal (auto-ocurrencia)
diag = np.diag(cooccurrence.values).copy()
diag[diag == 0] = 1  # evitar division por cero
cooc_norm = cooccurrence.values / np.sqrt(np.outer(diag, diag))
np.fill_diagonal(cooc_norm, 1)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cooc_norm, cmap='YlOrRd', aspect='auto', vmin=0, vmax=0.7)
ax.set_xticks(range(len(top20_names)))
ax.set_yticks(range(len(top20_names)))
ax.set_xticklabels(top20_names, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(top20_names, fontsize=9)
ax.set_title('Co-ocurrencia Normalizada — Top 20 Tags', fontweight='bold', fontsize=13)
plt.colorbar(im, ax=ax, shrink=0.8, label='Jaccard-like similarity')
plt.tight_layout()
plt.show()

# Pares con mayor co-ocurrencia
print('\nPares de tags con mayor co-ocurrencia:')
pairs = []
for i in range(len(top20_names)):
    for j in range(i+1, len(top20_names)):
        pairs.append((top20_names[i], top20_names[j], cooc_norm[i, j]))
pairs.sort(key=lambda x: -x[2])
for t1, t2, score in pairs[:10]:
    print(f'  {t1:20s} + {t2:20s}  →  {score:.3f}')

## 4. Categorización de Tags

Los tags de MTAT cubren múltiples dimensiones musicales:

In [ ]:
# ── Categorización manual de los top 50 tags ──
tag_categories = {
    'Instrumento': ['guitar', 'piano', 'drums', 'bass', 'strings', 'synth',
                    'organ', 'flute', 'violin', 'harpsichord', 'horn',
                    'sitar', 'harp', 'cello', 'saxophone', 'harmonica',
                    'oboe', 'trumpet', 'clarinet', 'banjo', 'mandolin'],
    'Voz': ['vocals', 'singer', 'female', 'male', 'vocal',
            'female vocal', 'male vocal', 'female voice', 'male voice',
            'choir', 'opera', 'singing', 'chanting', 'voice',
            'male singer', 'female singer', 'woman', 'man'],
    'Género': ['rock', 'pop', 'jazz', 'classical', 'metal', 'country',
               'folk', 'blues', 'punk', 'hip hop', 'reggae', 'soul',
               'funk', 'techno', 'dance', 'rap', 'electronica',
               'electronic', 'ambient', 'new age', 'world',
               'alternative', 'indie'],
    'Mood/Emoción': ['soft', 'hard', 'fast', 'slow', 'quiet', 'loud',
                     'calm', 'dark', 'happy', 'sad', 'aggressive',
                     'mellow', 'heavy', 'light', 'relaxing',
                     'energetic', 'upbeat', 'angry', 'romantic',
                     'melancholy', 'dreamy', 'chill'],
    'Época/Estilo': ['70s', '80s', '60s', '90s', 'classic',
                     'old', 'modern', 'new', 'experimental'],
    'Textura': ['beat', 'beats', 'no vocals', 'instrumental',
                'acoustic', 'electric', 'distortion', 'noise',
                'silence', 'echo', 'reverb'],
}

# Clasificar cada tag del top 50
cat_counts = {}
categorized_tags = {}

for tag in top50_tags.index:
    tag_lower = tag.lower().strip()
    found = False
    for category, keywords in tag_categories.items():
        if tag_lower in keywords:
            cat_counts[category] = cat_counts.get(category, 0) + 1
            categorized_tags[tag] = category
            found = True
            break
    if not found:
        cat_counts['Otros'] = cat_counts.get('Otros', 0) + 1
        categorized_tags[tag] = 'Otros'

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_colors_map = {
    'Instrumento': '#6C5CE7', 'Voz': '#FD79A8', 'Género': '#00B894',
    'Mood/Emoción': '#FDCB6E', 'Época/Estilo': '#74B9FF',
    'Textura': '#A29BFE', 'Otros': '#b2bec3'
}

cats = list(cat_counts.keys())
vals = [cat_counts[c] for c in cats]
cols = [cat_colors_map.get(c, '#b2bec3') for c in cats]

axes[0].pie(vals, labels=cats, colors=cols, autopct='%1.0f%%',
            pctdistance=0.85, startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Categorías de Tags (Top 50)', fontweight='bold')

# Clips por categoría (suma ponderada)
cat_clip_counts = {}
for tag, cat in categorized_tags.items():
    cat_clip_counts[cat] = cat_clip_counts.get(cat, 0) + int(tag_sums.get(tag, 0))

cats_sorted = sorted(cat_clip_counts.keys(), key=lambda c: -cat_clip_counts[c])
axes[1].bar(cats_sorted,
            [cat_clip_counts[c] for c in cats_sorted],
            color=[cat_colors_map.get(c, '#b2bec3') for c in cats_sorted],
            edgecolor='white')
axes[1].set_ylabel('Total Anotaciones (sum of tag=1)')
axes[1].set_title('Peso de cada Categoría (Top 50)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=30)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

print('Tags por categoría:')
for cat in cats_sorted:
    cat_tags = [t for t, c in categorized_tags.items() if c == cat]
    print(f'  {cat:15s} ({len(cat_tags):2d} tags): {", ".join(cat_tags[:8])}')

## 5. Metadata de Clips (`clip_info_final.csv`)

In [ ]:
clip_info = pd.read_csv(MTAT_DIR / 'clip_info_final.csv', sep='\t')

print(f'Dimensiones: {clip_info.shape}')
print(f'\nColumnas: {list(clip_info.columns)}')
print(f'\nDtypes:')
print(clip_info.dtypes.to_string())
print(f'\nNulls:')
print(clip_info.isnull().sum().to_string())

display(clip_info.head(10))

In [ ]:
# ── Estadísticas de artistas y álbumes ──
print('=== Artistas ===')
n_artists = clip_info['artist'].nunique()
print(f'Artistas únicos: {n_artists:,}')

artist_counts = clip_info['artist'].value_counts()
print(f'\nTop 15 artistas por # clips:')
for artist, count in artist_counts.head(15).items():
    bar = '█' * int(count / artist_counts.max() * 25)
    print(f'  {artist:30s} {count:>4} {bar}')

print(f'\n=== Álbumes ===')
if 'album' in clip_info.columns:
    n_albums = clip_info['album'].nunique()
    print(f'Álbumes únicos: {n_albums:,}')
elif 'title' in clip_info.columns:
    n_titles = clip_info['title'].nunique()
    print(f'Títulos únicos: {n_titles:,}')

In [ ]:
# ── Estructura de carpetas de audio ──
if len(mp3_files) > 0:
    # Las carpetas son 0-9 y a-f (hexadecimales)
    folders = set()
    for f in mp3_files:
        rel = f.relative_to(MTAT_AUDIO)
        if len(rel.parts) > 1:
            folders.add(rel.parts[0])
    
    print(f'Carpetas de audio: {sorted(folders)}')
    print(f'Total carpetas: {len(folders)}')
    
    # Clips por carpeta
    folder_counts = {}
    for f in mp3_files:
        rel = f.relative_to(MTAT_AUDIO)
        if len(rel.parts) > 1:
            folder = rel.parts[0]
            folder_counts[folder] = folder_counts.get(folder, 0) + 1
    
    print(f'\nClips por carpeta:')
    for folder in sorted(folder_counts.keys()):
        print(f'  {folder}: {folder_counts[folder]:,} clips')
else:
    print('⚠ No se encontraron archivos MP3')

## 6. Exploración de Audio

Visualizemos y escuchemos clips de diferentes categorías.

In [ ]:
def find_audio_path(mp3_path_str):
    """Find the actual audio file from the annotation path."""
    # annotations_final.csv tiene paths como: 'f/american_bach_soloists-j_s__bach...-01-...-0-29.mp3'
    p = MTAT_AUDIO / mp3_path_str
    if p.exists():
        return p
    # Try without leading folder
    p2 = MTAT_AUDIO / 'mp3' / mp3_path_str
    if p2.exists():
        return p2
    return None


def load_audio(path, sr=SR, duration=None):
    """Load audio safely."""
    try:
        y, _ = librosa.load(str(path), sr=sr, mono=True, duration=duration)
        return y
    except Exception as e:
        print(f'Error loading {path}: {e}')
        return None


# ── Verificar que podemos cargar audio ──
# Find first valid path
sample_paths = annotations['mp3_path'].head(20).tolist()
valid_path = None
for sp in sample_paths:
    fp = find_audio_path(sp)
    if fp is not None:
        valid_path = fp
        break

if valid_path:
    print(f'✓ Audio path encontrado: {valid_path}')
    y = load_audio(valid_path, duration=10)
    if y is not None:
        print(f'✓ Audio cargado: {len(y)} samples ({len(y)/SR:.1f}s)')
else:
    print('⚠ No se pudo encontrar un archivo de audio válido.')
    print('  Paths en CSV:', sample_paths[:3])
    print('  MTAT_AUDIO:', MTAT_AUDIO)
    if mp3_files:
        print('  Ejemplo MP3:', mp3_files[0])

In [ ]:
# ── Escuchar muestras por tag ──
selected_tags = ['guitar', 'piano', 'female vocal', 'drums', 'classical',
                 'rock', 'electronic', 'ambient']

# Filtrar a tags que existan en el dataset
available_tags = [t for t in selected_tags if t in tag_columns]
if not available_tags:
    # Fallback: usar top tags
    available_tags = top50_tags.index[:8].tolist()

print(f'🎧 Muestras de Audio por Tag (primeros 10 segundos)\n')

for tag in available_tags:
    # Find a clip with this tag
    tag_clips = annotations[annotations[tag] == 1]
    if len(tag_clips) == 0:
        continue
    
    found = False
    for _, row in tag_clips.head(10).iterrows():
        audio_path = find_audio_path(row['mp3_path'])
        if audio_path is not None:
            y = load_audio(audio_path, duration=10)
            if y is not None:
                # Get all tags for this clip
                clip_tags = [t for t in tag_columns if row.get(t, 0) == 1]
                display(HTML(
                    f'<b>🏷 {tag}</b> — <i>{row["mp3_path"]}</i>'
                    f'<br><small>Tags: {", ".join(clip_tags[:10])}</small>'
                ))
                display(Audio(data=y, rate=SR))
                print()
                found = True
                break
    
    if not found:
        print(f'  {tag}: no se encontró audio')

In [ ]:
# ── Waveforms y Mel Spectrograms por tag ──
display_tags = available_tags[:6]  # Limitar a 6

fig, axes = plt.subplots(len(display_tags), 2, figsize=(16, len(display_tags) * 2.5))
if len(display_tags) == 1:
    axes = axes.reshape(1, -1)

tag_colors = plt.cm.Set2(np.linspace(0, 1, len(display_tags)))

for i, tag in enumerate(display_tags):
    tag_clips = annotations[annotations[tag] == 1]
    y = None
    for _, row in tag_clips.head(10).iterrows():
        audio_path = find_audio_path(row['mp3_path'])
        if audio_path:
            y = load_audio(audio_path, duration=10)
            if y is not None:
                break
    
    if y is None:
        axes[i, 0].text(0.5, 0.5, 'No audio', ha='center', va='center')
        axes[i, 1].text(0.5, 0.5, 'No audio', ha='center', va='center')
        continue
    
    # Waveform
    times = np.arange(len(y)) / SR
    axes[i, 0].plot(times, y, linewidth=0.3, color=tag_colors[i])
    axes[i, 0].set_ylabel(tag, fontweight='bold', fontsize=10)
    axes[i, 0].set_xlim(0, min(10, len(y)/SR))
    axes[i, 0].set_ylim(-1, 1)
    if i == 0:
        axes[i, 0].set_title('Waveform', fontweight='bold')
    if i < len(display_tags) - 1:
        axes[i, 0].set_xticklabels([])
    else:
        axes[i, 0].set_xlabel('Tiempo (s)')
    
    # Mel Spectrogram
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=64, fmax=8000)
    S_dB = librosa.power_to_db(S, ref=np.max)
    librosa.display.specshow(S_dB, sr=SR, x_axis='time', y_axis='mel',
                            ax=axes[i, 1], cmap='magma', fmax=8000)
    if i == 0:
        axes[i, 1].set_title('Mel Spectrogram', fontweight='bold')
    axes[i, 1].set_ylabel('')
    if i < len(display_tags) - 1:
        axes[i, 1].set_xlabel('')

plt.suptitle('Waveforms y Espectrogramas por Tag — MTAT',
             fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 7. Splits de Entrenamiento

El paper SemiSupCon usa el split canónico **12:1:3** basado en las carpetas del dataset.

In [ ]:
# ── Simular split canónico 12:1:3 basado en carpetas ──
# Carpetas 0-b (12) = train, c (1) = val, d-f (3) = test

def get_folder_from_path(mp3_path):
    """Extract first folder character from path."""
    parts = str(mp3_path).split('/')
    return parts[0] if len(parts) > 1 else '?'

annotations['folder'] = annotations['mp3_path'].apply(get_folder_from_path)

train_folders = list('0123456789ab')
val_folders = ['c']
test_folders = ['d', 'e', 'f']

train_mask = annotations['folder'].isin(train_folders)
val_mask = annotations['folder'].isin(val_folders)
test_mask = annotations['folder'].isin(test_folders)

n_train = train_mask.sum()
n_val = val_mask.sum()
n_test = test_mask.sum()

print('=== Split Canónico 12:1:3 ===')
print(f'Train (carpetas 0-b): {n_train:>6,} clips ({n_train/len(annotations)*100:.1f}%)')
print(f'Val   (carpeta c):    {n_val:>6,} clips ({n_val/len(annotations)*100:.1f}%)')
print(f'Test  (carpetas d-f): {n_test:>6,} clips ({n_test/len(annotations)*100:.1f}%)')
print(f'Total:                {n_train+n_val+n_test:>6,}')

# Verificar distribución de tags por split
print(f'\nDistribución del tag "guitar" por split:')
if 'guitar' in tag_columns:
    print(f'  Train: {annotations.loc[train_mask, "guitar"].sum():>5,} clips')
    print(f'  Val:   {annotations.loc[val_mask, "guitar"].sum():>5,} clips')
    print(f'  Test:  {annotations.loc[test_mask, "guitar"].sum():>5,} clips')

# Visualizar
fig, ax = plt.subplots(figsize=(8, 4))
split_data = [n_train, n_val, n_test]
split_labels = [f'Train\n{n_train:,}', f'Val\n{n_val:,}', f'Test\n{n_test:,}']
split_colors = ['#6C5CE7', '#FDCB6E', '#E17055']
ax.bar(split_labels, split_data, color=split_colors, edgecolor='white', width=0.5)
ax.set_ylabel('Número de Clips')
ax.set_title('Split Train/Val/Test — MTAT', fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

## 8. Relevancia para SemiSupCon

¿Cómo usa SemiSupCon este dataset?

In [ ]:
# ── Simulación: cómo se construye la contrastive matrix M ──
# En SemiSupCon, dos muestras supervisadas son "positivas" si comparten
# al menos C tags en común (por defecto C=1)

import random
random.seed(42)

# Tomar un mini-batch de 16 muestras supervisadas
top50_list = top50_tags.index.tolist()
batch_df = annotations[annotations[top50_list].sum(axis=1) > 0].sample(16, random_state=42)
batch_tags = batch_df[top50_list].values  # (16, 50)

# Construir contrastive matrix con C=1 (al menos 1 tag en común)
N = len(batch_tags)
M = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i != j:
            common = np.sum(batch_tags[i] * batch_tags[j])
            M[i, j] = 1 if common >= 1 else 0

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Binary matrix
axes[0].imshow(M, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[0].set_title(f'Contrastive Matrix M (C=1)\n{int(M.sum())} positivos / {N*(N-1)} pares', fontweight='bold')
axes[0].set_xlabel('Sample j')
axes[0].set_ylabel('Sample i')

# Continuous similarity (semantic weighing)
alpha = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        if i != j:
            common = np.sum(batch_tags[i] * batch_tags[j])
            Ci = np.sum(batch_tags[i])
            Cj = np.sum(batch_tags[j])
            if Ci + Cj > 0:
                alpha[i, j] = 2 * common / (Ci + Cj)  # Dice coefficient

im = axes[1].imshow(alpha, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
axes[1].set_title('Semantic Similarity α\n(Dice coefficient)', fontweight='bold')
axes[1].set_xlabel('Sample j')
axes[1].set_ylabel('Sample i')
plt.colorbar(im, ax=axes[1], shrink=0.8)

plt.suptitle('Cómo SemiSupCon usa los tags de MTAT para construir positivos',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

sparsity = M.sum() / (N * (N-1))
print(f'\nMatrix sparsity (proporción de positivos): {sparsity:.3f}')
print(f'Esto significa que en un batch de {N} muestras, {sparsity*100:.1f}% de los pares son positivos')

In [ ]:
# ── Resumen final ──
print('╔══════════════════════════════════════════════════════════════╗')
print('║           RESUMEN: MagnaTagATune (MTAT)                    ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  Clips totales:       {len(annotations):>6,}                              ║')
print(f'║  Tags totales:        {len(tag_columns):>6}                              ║')
print(f'║  Tags usados (top):       50 (estándar MIR)               ║')
print(f'║  Duración/clip:       ~29 segundos                        ║')
print(f'║  Sample rate:         16,000 Hz (original)                ║')
print(f'║  Formato:             MP3                                 ║')
print(f'║  Artistas únicos:     {n_artists:>6,}                              ║')
print(f'║  Tags/clip (media):   {tags_per_clip.mean():>6.1f}                              ║')
print(f'╠══════════════════════════════════════════════════════════════╣')
print(f'║  SPLIT CANÓNICO                                           ║')
print(f'║  Train:               {n_train:>6,} clips (carpetas 0-b)         ║')
print(f'║  Val:                 {n_val:>6,} clips (carpeta c)            ║')
print(f'║  Test:                {n_test:>6,} clips (carpetas d-f)         ║')
print(f'╠══════════════════════════════════════════════════════════════╣')
print(f'║  USO EN SEMISUPCON                                        ║')
print(f'║  Rol: Dataset supervisado S (con labels)                  ║')
print(f'║  Tags → Contrastive Matrix M                             ║')
print(f'║  C=1: positivo si comparten ≥1 tag                       ║')
print(f'║  Se combina con FMA Medium (U, sin labels)               ║')
print(f'╚══════════════════════════════════════════════════════════════╝')